# RLSSM — Basic tutorial
This tutorial is a first pass through an RLSSM workflow in HSSM. The goal is to
get from a preset model to a fitted hierarchical analysis without stopping on
every internal object along the way.
By the end you will have:
- simulated a synthetic RLSSM dataset with [`ssm-simulators`](https://github.com/lnccbrown/ssm-simulators) (`ssms.rl`),
- fit a **hierarchical** (multi-participant) RLSSM in HSSM from the named preset `2AB_RW_Angle`,
- checked a simple recovery summary, and
- run a **posterior predictive check** tailored to RLSSMs.
> **Where this sits in the suite:** this is the entry point. Later tutorials build
> *custom* learning/decision models
> ([Custom models with ssms.rl](rlssm_advanced.ipynb) ·
> [Restless learner](rlssm_restless_learner.ipynb)) and show HSSM-native
> registration ([Registering custom models in HSSM](rlssm_hssm_custom_models.ipynb)).


## 1. The core idea
### 1.1 The learning process: Rescorla–Wagner
Our task is a **two-armed bandit**: on each trial the participant picks one of two
options and receives binary feedback (reward = 1, no reward = 0). The participant
keeps a running value estimate $Q$ for each option and updates it using the
**Rescorla–Wagner delta rule**. After choosing option $c$ and observing reward $r$:
$$
Q_{c} \leftarrow Q_{c} + \alpha \, \underbrace{(r - Q_{c})}_{\text{prediction error}}
$$
- $Q_c$ is the current value estimate for the chosen option.
- $r - Q_c$ is the **reward prediction error** — how surprising the outcome was.
- $\alpha \in (0, 1)$ is the **learning rate** (`rl_alpha`): large $\alpha$ means the
  participant updates quickly and weights recent outcomes heavily; small $\alpha$
  means slow, stable learning. Only the chosen option's value is updated.
### 1.2 Coupling value to the decision: the drift rate
On each trial, the *difference* in learned value between the two options sets how
strongly evidence flows toward one option during the decision. Concretely the
**drift rate** $v$ of the decision process is
$$
v = \big(Q_{1} - Q_{0}\big)\cdot \texttt{scaler},
$$
where `scaler` converts a value difference into drift units. When the two options
look equally good ($Q_1 \approx Q_0$) drift is near zero and choices are slow and
near-chance; as learning separates the values, $|v|$ grows and choices become faster
and more consistent. **`v` is not a free parameter you estimate directly — it is
*computed* from the learning process on every trial.** This is the heart of an RLSSM.
### 1.3 The decision process: the angle SSM (brief)
Given a drift rate, the decision itself is produced by a **sequential sampling
model**. If you have seen the drift-diffusion model (DDM) before, this will be
familiar: noisy evidence accumulates from a starting point until it hits one of two
boundaries, and *which* boundary and *when* determine the choice and RT. We use the
**angle** variant, whose only addition to the standard DDM is a **linearly
collapsing boundary** — the decision threshold narrows over time, which helps capture
fast errors often seen in speeded choice. Its parameters are `a`, `z`, `t`, and
`theta`, while `v` is supplied by the learning process.
### 1.4 Why hierarchical?
We usually have many participants, each slightly different. A **hierarchical** model
estimates a group-level value for each parameter **and** a per-participant deviation
from it, letting participants share statistical strength ("partial pooling"). In
this tutorial we will use one reusable prior template for all free parameters rather
than introducing every modeling detail separately.


## 2. Setup
We import HSSM and the `ssms.rl` simulation API, then set two global options that
matter for RLSSM work:
- **`hssm.set_floatX("float32")`** — RLSSM likelihoods run through a JAX scan over
  trials; single precision keeps this fast and is what the RLSSM pipeline is tuned
  for.
- We silence a few noisy library warnings so the notebook output stays readable.


In [ ]:
import logging
import warnings
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ssms import rl
import hssm
warnings.filterwarnings("ignore")
logging.getLogger("jax._src.xla_bridge").setLevel(logging.ERROR)
hssm.set_floatX("float32", update_jax=True)
RANDOM_SEED = 20260704


### Run size
This tutorial uses a quick configuration so readers can complete a full
end-to-end RLSSM workflow without long waits.
**Optional longer run (advanced):** after your first pass, increase the values
in the next cell (participants, trials, tune, and draws) to get tighter
recovery and PPC curves.


In [ ]:
N_PARTICIPANTS = 5
N_TRIALS = 70
N_CHAINS = 2
N_TUNE = 300
N_DRAWS = 300

print(
    f"quick mode | participants={N_PARTICIPANTS} trials={N_TRIALS} "
    f"tune={N_TUNE} draws={N_DRAWS}"
)


## 3. Pick a model: the `2AB_RW_Angle` preset
`ssms.rl` ships **presets** that bundle a task environment, a learning rule, and a
decision process into one ready-to-use model. `2AB_RW_Angle` is exactly the model we
described in above: a **2**-**a**rmed **b**andit with **R**escorla–**W**agner learning
and an **angle** decision process. `rl.preset.info(...)` prints a readable summary of
everything the preset contains.


In [ ]:
print(rl.preset.info("2AB_RW_Angle"))

To fit this preset in HSSM, there are only two facts you need right away:
- the free parameters are `rl_alpha`, `scaler`, `a`, `z`, `t`, and `theta`, and
- the drift `v` is **computed** from the learning process rather than sampled.

We confirm that below.



In [ ]:
ssms_config = rl.preset.get("2AB_RW_Angle")
assembled = ssms_config.assemble(backend="jax")
print("computed params (driven by the learner):", assembled.computed_params)
assert "v" in assembled.computed_params


## 4. Define ground-truth parameters
Because this is a tutorial, we *simulate* data from known parameters so we can later
check whether the model recovers them. We choose a **group mean** for each parameter
and give each participant a small random deviation around that mean, so participants
genuinely differ — that is what the hierarchical model will try to recover.
`SDS` sets how spread out participants are for each parameter (bigger = more
individual variability), and `BOUNDS` keeps every sampled value inside the range the
model supports.


In [ ]:
GROUP_THETA = {
    "rl_alpha": 0.08,  # learning rate (small -> gradual, visible learning)
    "scaler": 2.5,  # value-difference -> drift gain
    "a": 1.2,  # boundary separation
    "z": 0.5,  # starting-point bias (0.5 = unbiased)
    "t": 0.25,  # non-decision time (s)
    "theta": 0.35,  # boundary collapse angle
}
# Between-participant SD for each parameter (individual differences to recover).
SDS = {
    "rl_alpha": 0.03,
    "scaler": 0.40,
    "a": 0.20,
    "z": 0.06,
    "t": 0.05,
    "theta": 0.10,
}
# Keep sampled values inside supported ranges.
BOUNDS = {
    "rl_alpha": (0.01, 1.0),
    "scaler": (0.1, 5.0),
    "a": (0.3, 2.5),
    "z": (0.1, 0.9),
    "t": (0.05, 1.0),
    "theta": (0.0, 1.2),
}
LIST_PARAMS = list(GROUP_THETA);

rng = np.random.default_rng(RANDOM_SEED)
theta_arrays = {
    name: np.clip(
        rng.normal(GROUP_THETA[name], SDS[name], N_PARTICIPANTS), *BOUNDS[name]
    )
    for name in LIST_PARAMS
}

# One row per participant: their true parameter values (for the recovery check later).
true_params = pd.DataFrame(theta_arrays)
true_params.index.name = "participant_id"
true_params.round(3)

## 5. Simulate the data
`rl.Simulator(config).simulate(...)` runs the full generative loop for every
participant and trial: compute the drift from current Q-values → run the angle SSM to
get a choice and RT → deliver feedback → update the Q-values. Passing **arrays** of
parameters (one value per participant) produces a balanced multi-participant panel.


In [ ]:
data = rl.Simulator(ssms_config).simulate(
    theta=theta_arrays,
    n_trials=N_TRIALS,
    n_participants=N_PARTICIPANTS,
    random_state=RANDOM_SEED,
)
# Validate the panel matches what the model expects before doing anything else.
ssms_config.validate_data(data).raise_for_errors()

print("rows:", len(data), "| columns:", list(data.columns))
data.head()

Each row is one trial. The columns we care about:
- `participant_id`, `trial_id`: who and when.
- `response`: the chosen arm (-1 or 1).
- `rt`: response time in seconds.
- `feedback`: reward (0 or 1), used by the learner.

The next cell is a quick sanity check that learning is visible in the simulated data.
It does three things:
1. bins trials into windows of 10;
2. computes choice accuracy as P(chose high-reward arm);
3. computes mean RT per bin.

Expected pattern: accuracy should rise above chance, and RT should decrease as values separate.


In [ ]:
BIN = 10
learn = data[data["rt"] > 0].copy()
learn["chose_high"] = (learn["response"] == -1).astype(float)  # -1 == high-reward arm
learn["trial_bin"] = (learn["trial_id"] // BIN) * BIN
acc_curve = learn.groupby("trial_bin")["chose_high"].mean()
rt_curve = learn.groupby("trial_bin")["rt"].mean()
centers = acc_curve.index + BIN / 2

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(centers, acc_curve.values, "o-", color="tab:green")
axes[0].axhline(0.5, color="0.7", ls="--", lw=1, label="chance")
axes[0].set(
    xlabel="Trial",
    ylabel="P(chose high-reward arm)",
    title="Accuracy: choices shift to the good arm",
    ylim=(0, 1),
)
axes[0].legend(frameon=False)

axes[1].plot(centers, rt_curve.values, "o-", color="tab:purple")
axes[1].set(xlabel="Trial", ylabel="Mean RT (s)", title="Speed: responses get faster")
fig.suptitle("Learning curves (simulated data)")
plt.show()

## 6. Build the HSSM model


To keep this first fit readable, we use one small helper that applies the same
hierarchical template to every free parameter: one group mean plus participant-level
deviations. Later tutorials show how to customize these priors in more detail.


In [ ]:
# Prior on the per-participant deviations: mean 0, with a learned spread (sigma).
PARTICIPANT_EFFECT_PRIOR = {
    "name": "Normal",
    "mu": 0,
    "sigma": {"name": "HalfNormal", "sigma": 0.5},
}


def hierarchical_param(name, lower, upper, mu, sigma):
    """Build a group intercept (TruncatedNormal) + per-participant random effect."""
    return hssm.Param(
        name,
        formula=f"{name} ~ 1 + (1|participant_id)",
        prior={
            "Intercept": hssm.Prior(
                "TruncatedNormal", lower=lower, upper=upper, mu=mu, sigma=sigma
            ),
            "1|participant_id": PARTICIPANT_EFFECT_PRIOR,
        },
    )


A few arguments deserve a quick note:
- **`model="2AB_RW_Angle"`** tells HSSM to use the same preset we simulated from.
- **`include=[...]`** applies the same hierarchical prior template to each free parameter.
- **`p_outlier=0` / `lapse=None`** turn off the outlier/lapse mixture for this first fit.
- **`process_initvals=False`** is the main RLSSM-specific setting to remember.


In [ ]:
model = hssm.RLSSM(
    data=data,
    model="2AB_RW_Angle",
    p_outlier=0,
    lapse=None,
    process_initvals=False,
    include=[
        hierarchical_param("rl_alpha", 0.01, 1.0, 0.15, 0.15),
        hierarchical_param("scaler", 0.1, 5.0, 2.0, 0.8),
        hierarchical_param("a", 0.3, 2.5, 1.1, 0.3),
        hierarchical_param("z", 0.1, 0.9, 0.5, 0.15),
        hierarchical_param("t", 0.05, 1.0, 0.25, 0.1),
        hierarchical_param("theta", 0.0, 1.2, 0.35, 0.15),
    ],
)

print("participants:", model.n_participants, "| trials/participant:", model.n_trials)
print("free parameters:", list(model.params.keys()))
assert model.model_name == "2AB_RW_Angle"
assert "rl_alpha" in model.params
assert "v" not in model.params  # computed by the learner, never sampled


## 7. Sample the posterior
We now draw from the posterior. This is the step that learns the parameters from data. In this quick tutorial
setup, sampling is intentionally short.


In [ ]:
idata = model.sample(
    sampler="numpyro",
    draws=N_DRAWS,
    tune=N_TUNE,
    chains=N_CHAINS,
    cores=N_CHAINS,
    target_accept=0.9,
    random_seed=RANDOM_SEED,
)
idata

## 8. A quick recovery check
The first question after fitting is simple: did the model recover the group-level
values we simulated from? We start with that group-level check here. Participant-by-
participant recovery is useful too, but it is easier to read once the basic workflow
feels familiar.


In [ ]:
def group_recovery(idata, true_group):
    """Group intercept posterior vs. true group mean, per parameter."""
    names = [f"{p}_Intercept" for p in LIST_PARAMS]
    summ = az.summary(
        idata,
        var_names=names,
        kind="stats",
        ci_kind="hdi",
        ci_prob=0.94,
        round_to="none",
    )
    summ.index = LIST_PARAMS
    summ["true"] = [true_group[p] for p in LIST_PARAMS]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    y = np.arange(len(LIST_PARAMS))
    ax.errorbar(
        summ["mean"],
        y,
        xerr=[summ["mean"] - summ["hdi94_lb"], summ["hdi94_ub"] - summ["mean"]],
        fmt="o",
        capsize=4,
        label="posterior (94% HDI)",
    )
    ax.scatter(
        summ["true"], y, color="crimson", marker="D", zorder=5, label="true group mean"
    )
    ax.set_yticks(y)
    ax.set_yticklabels(LIST_PARAMS)
    ax.invert_yaxis()
    ax.set_title("Group-level recovery")
    ax.legend()
    fig.tight_layout()
    plt.show()
    return summ


### Group-level recovery
Each posterior interval (blue) should sit reasonably close to the corresponding true
group mean (red diamond).


In [ ]:
group_summary = group_recovery(idata, GROUP_THETA)
group_summary[["mean", "hdi94_lb", "hdi94_ub", "true"]].round(3)

## 9. Posterior predictive check (RLSSM-aware)
A posterior predictive check (PPC) asks: *if we simulate new data from the fitted
parameters, does it look like the data we observed?* For an RLSSM there is one extra
wrinkle: the predicted behavior depends on the learning trajectory. The helper below
handles that bookkeeping by replaying each participant's observed reward history
during PPC simulation, then comparing the predicted learning curve and RT distribution
with the observed data.


In [ ]:
def plot_rl_ppc(idata, observed_data, ssms_config, *, n_participants, random_seed):
    """Simulate an RL-aware PPC and plot the main checks."""
    def draw_posterior_theta(draw_idx):
        posterior = idata.posterior
        if hasattr(posterior, "to_dataset"):
            posterior = posterior.to_dataset()  # PyMC 6 returns a DataTree node
        post = posterior.stack(sample=("chain", "draw"))
        theta = {}
        for name in LIST_PARAMS:
            re = post[f"{name}_1|participant_id"]
            pid_dim = [d for d in re.dims if d not in ("sample",)][0]
            vals = (post[f"{name}_Intercept"] + re).isel(sample=draw_idx)
            ids = [int(v) for v in re[pid_dim].values]
            series = pd.Series(np.asarray(vals.values), index=ids).sort_index()
            theta[name] = series.reindex(range(n_participants)).to_numpy()
        return theta
    def learning_curve(df, bin_size=10):
        d = df[df["rt"] > -900].copy()
        d["chose_high"] = (d["response"] == -1).astype(float)
        d["trial_bin"] = (d["trial_id"] // bin_size) * bin_size
        return d.groupby("trial_bin")["chose_high"].mean()
    def signed_rt(df):
        d = df[df["rt"] > -900].copy()
        return np.where(
            d["response"].astype(int) == -1,
            -d["rt"].astype(float),
            d["rt"].astype(float),
        )
    n_ppc_draws = 8
    n_samples = idata.posterior.sizes["chain"] * idata.posterior.sizes["draw"]
    ppc_rng = np.random.default_rng(random_seed + 1)
    draw_ids = ppc_rng.choice(
        n_samples, size=min(n_ppc_draws, n_samples), replace=False
    )
    ppc_frames = []
    for offset, draw_id in enumerate(draw_ids):
        theta_d = draw_posterior_theta(int(draw_id))
        ppc_draw = rl.Simulator(ssms_config).simulate(
            theta=theta_d,
            mode="ppc",
            observed_data=observed_data,
            random_state=random_seed + 100 + offset,
        )
        ppc_draw["ppc_draw"] = offset
        ppc_frames.append(ppc_draw)
    ppc_data = pd.concat(ppc_frames, ignore_index=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
    obs_curve = learning_curve(observed_data)
    ppc_curves = pd.concat(
        [learning_curve(group).rename(draw) for draw, group in ppc_data.groupby("ppc_draw")],
        axis=1,
    ).sort_index()
    centers = obs_curve.index + 5
    axes[0].fill_between(
        ppc_curves.index + 5,
        ppc_curves.quantile(0.03, axis=1),
        ppc_curves.quantile(0.97, axis=1),
        alpha=0.25,
        color="tab:blue",
        label="PPC 94% band",
    )
    axes[0].plot(
        ppc_curves.index + 5,
        ppc_curves.mean(axis=1),
        color="tab:blue",
        lw=1.5,
        label="PPC mean",
    )
    axes[0].plot(centers, obs_curve.values, "o-", color="black", label="observed")
    axes[0].axhline(0.5, color="0.7", ls="--", lw=1)
    axes[0].set(
        xlabel="Trial",
        ylabel="P(chose high-reward arm)",
        title="Learning-curve PPC",
        ylim=(0, 1),
    )
    axes[0].legend(frameon=False)
    axes[1].hist(
        signed_rt(observed_data),
        bins=40,
        density=True,
        histtype="step",
        lw=1.8,
        color="black",
        label="observed",
    )
    axes[1].hist(
        signed_rt(ppc_data),
        bins=40,
        density=True,
        histtype="step",
        lw=1.8,
        color="tab:blue",
        label="PPC",
    )
    axes[1].axvline(0, color="0.6", lw=1)
    axes[1].set(
        xlabel="Signed RT (negative = high-reward choice)",
        ylabel="density",
        title="Signed-RT PPC",
    )
    axes[1].legend(frameon=False)
    plt.show()
    print("PPC datasets:", len(draw_ids), "| total rows:", len(ppc_data))
    return ppc_data


In [ ]:
ppc_data = plot_rl_ppc(
    idata,
    observed_data=data,
    ssms_config=ssms_config,
    n_participants=N_PARTICIPANTS,
    random_seed=RANDOM_SEED,
)


## 10. Summary
You have run a complete beginner-friendly RLSSM workflow:
1. **Chose a model** — the `2AB_RW_Angle` preset (Rescorla–Wagner learning + angle SSM).
2. **Simulated** a hierarchical dataset from known parameters and confirmed learning.
3. **Fit** the same preset directly in HSSM with a hierarchical prior template.
4. **Sampled** the posterior with NumPyro using `process_initvals=False`.
5. **Checked** one recovery summary and an RL-aware PPC.
### Where to go next
- **[Custom models with ssms.rl](rlssm_advanced.ipynb)** — build your own task
  environment and learning rule instead of using a preset.
- **[Restless learner](rlssm_restless_learner.ipynb)** — one learner driving *several*
  decision parameters at once.
- **[Registering custom models in HSSM](rlssm_hssm_custom_models.ipynb)** — the
  HSSM-native registry path.
> **Note:** HSSM also supports *choice-only* reinforcement-learning models (no RT).
> Those are documented separately once fully validated against the current release.
